# **1. Perkenalan Dataset**


## Wine Quality Dataset

Dataset yang digunakan adalah **Wine Quality Dataset** dari UCI Machine Learning Repository.

Dataset ini berisi informasi physicochemical dari sampel wine merah (red wine) beserta skor kualitasnya.

**Sumber:** https://archive.ics.uci.edu/ml/datasets/wine+quality

**Fitur dataset:**
- `fixed acidity` – Kadar asam tetap
- `volatile acidity` – Kadar asam yang mudah menguap
- `citric acid` – Kadar asam sitrat
- `residual sugar` – Sisa gula setelah fermentasi
- `chlorides` – Kadar garam klorida
- `free sulfur dioxide` – Kadar SO2 bebas
- `total sulfur dioxide` – Total SO2
- `density` – Kepadatan wine
- `pH` – Tingkat keasaman
- `sulphates` – Kadar sulfat
- `alcohol` – Kadar alkohol
- `quality` – **Target**: Skor kualitas (0-10), akan diubah menjadi klasifikasi biner (good/bad)

**Tipe masalah:** Binary Classification (kualitas wine: good atau bad)

# **2. Import Library**

Pada tahap ini, kita mengimpor semua pustaka Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Utilities
import warnings
import os
warnings.filterwarnings('ignore')

# Set style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('Library berhasil diimport!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

# **3. Memuat Dataset**

Dataset Wine Quality dimuat dari file CSV. Dataset ini merupakan data tabular yang berisi 1599 sampel red wine dengan 12 fitur.

In [ ]:
# Memuat dataset
df = pd.read_csv('../winequality-red.csv', sep=';')

print('=== INFORMASI DATASET ===')
print(f'Jumlah baris   : {df.shape[0]}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'Nama kolom     : {list(df.columns)}')
print()
print('=== 5 DATA PERTAMA ===')
df.head()

In [ ]:
# Informasi tipe data
print('=== INFO TIPE DATA ===')
df.info()

In [ ]:
# Statistik deskriptif
print('=== STATISTIK DESKRIPTIF ===')
df.describe().T

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# 4.1 Cek Missing Values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing (%)': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0 else 'Tidak ada missing values!')

In [ ]:
# 4.2 Cek Duplikat
print('=== DATA DUPLIKAT ===')
duplicates = df.duplicated().sum()
print(f'Jumlah duplikat: {duplicates}')

In [ ]:
# 4.3 Distribusi Target Variable (quality)
print('=== DISTRIBUSI TARGET (quality) ===')
print(df['quality'].value_counts())
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram distribusi quality
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribusi Skor Quality Wine', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Jumlah Sampel')
axes[0].tick_params(rotation=0)

# Pie chart
df['quality'].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Proporsi Quality Score', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('distribusi_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot distribusi target disimpan!')

In [ ]:
# 4.4 Distribusi Semua Fitur
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribusi: {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Seluruh Fitur Dataset', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distribusi_fitur.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot distribusi fitur disimpan!')

In [ ]:
# 4.5 Heatmap Korelasi
fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.5
)
ax.set_title('Heatmap Korelasi Antar Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap korelasi disimpan!')

In [ ]:
# 4.6 Boxplot untuk Deteksi Outlier
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[i].set_title(f'Boxplot: {col}', fontweight='bold')
    axes[i].set_xlabel(col)

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplot Deteksi Outlier', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplot_outlier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Boxplot outlier disimpan!')

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing dilakukan untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Tahapan preprocessing yang dilakukan:
1. Binarisasi target (quality → good/bad)
2. Menghapus data duplikat
3. Menangani outlier dengan IQR method
4. Feature-Target split
5. Train-Test split
6. Standarisasi fitur (StandardScaler)
7. Simpan hasil preprocessing

In [ ]:
# 5.1 Binarisasi Target: quality >= 6 → 'good' (1), else → 'bad' (0)
print('=== SEBELUM BINARISASI ===')
print(df['quality'].value_counts())

df['quality'] = (df['quality'] >= 6).astype(int)

print()
print('=== SETELAH BINARISASI ===')
print(df['quality'].value_counts())
print(f"\nLabel: 1 = Good wine (quality >= 6), 0 = Bad wine (quality < 6)")

In [ ]:
# 5.2 Hapus Duplikat
print(f'Jumlah baris sebelum hapus duplikat: {len(df)}')
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f'Jumlah baris setelah hapus duplikat : {len(df)}')

In [ ]:
# 5.3 Penanganan Outlier dengan IQR Method
feature_cols = [c for c in df.columns if c != 'quality']

def remove_outliers_iqr(df, cols, factor=1.5):
    df_clean = df.copy()
    for col in cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        before = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
        removed = before - len(df_clean)
        if removed > 0:
            print(f'  [{col}] Removed {removed} outliers (range: {lower:.3f} - {upper:.3f})')
    return df_clean.reset_index(drop=True)

print(f'Jumlah baris sebelum hapus outlier: {len(df)}')
df = remove_outliers_iqr(df, feature_cols, factor=3.0)
print(f'Jumlah baris setelah hapus outlier : {len(df)}')

In [ ]:
# 5.4 Feature-Target Split
X = df.drop('quality', axis=1)
y = df['quality']

print(f'Fitur (X) shape : {X.shape}')
print(f'Target (y) shape: {y.shape}')
print(f'Fitur: {list(X.columns)}')
print(f'Distribusi target:\n{y.value_counts()}')

In [ ]:
# 5.5 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('=== TRAIN-TEST SPLIT ===')
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}  | y_test : {y_test.shape}')
print()
print(f'Distribusi train:\n{y_train.value_counts()}')
print()
print(f'Distribusi test:\n{y_test.value_counts()}')

In [ ]:
# 5.6 Standarisasi Fitur
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

print('=== HASIL STANDARISASI ===')
print(f'Mean X_train (before): {X_train.mean().round(3).to_dict()}')
print()
print(f'Mean X_train (after) : {X_train_scaled.mean().round(3).to_dict()}')
print(f'Std  X_train (after) : {X_train_scaled.std().round(3).to_dict()}')

In [ ]:
# 5.7 Simpan Hasil Preprocessing
output_dir = 'winequality_preprocessing'
os.makedirs(output_dir, exist_ok=True)

X_train_scaled.to_csv(f'{output_dir}/X_train.csv', index=False)
X_test_scaled.to_csv(f'{output_dir}/X_test.csv', index=False)
y_train.reset_index(drop=True).to_csv(f'{output_dir}/y_train.csv', index=False)
y_test.reset_index(drop=True).to_csv(f'{output_dir}/y_test.csv', index=False)

print('=== DATA PREPROCESSING TERSIMPAN ===')
for f in os.listdir(output_dir):
    size = os.path.getsize(f'{output_dir}/{f}')
    print(f'  {f}: {size:,} bytes')

print()
print('Preprocessing selesai! Data siap untuk pelatihan model.')